# ドローン群を毎秒 QUBO で動かす（トラブル時はその場で組み直し）

- 地域データは **任意の JSON**（`data_transformed.json` と同じ形式）を使えます
- 1 秒ごとに、飛行中の全機の「次の行き先」を QUBO で決め直します
- 飛行中に **危険区域・要救助者を発見**したとき、**機体の故障・バッテリー異常**が起きたときは、その場で QUBO を組み直します
- **各市町村に複数のドローンを配置**します。基地（発着・充電地点）は候補地点から**配置 QUBO** で選び、15 分ごとと機体が減ったときに見直します
- 地域を 3 km 四方の**巡回ブロック**に分け、**一定時間（既定 60 分）通っていないブロックを巡回し直す**ように動きます
- 見える範囲は 150〜500 m、機体は 20 機、バッテリーは 15 分飛行・満充電 45 分（シナリオで変更可）

# 準備

In [ ]:
!pip install -q amplify dimod dwave-neal
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!git clone -q -b claude/request-5v4fsc https://github.com/mori560269/Q-Questgrow-.git
%cd Q-Questgrow-
import matplotlib.font_manager as fm
try:
    fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf')
except Exception:
    pass

Fixstars Amplify AE で解く場合はトークンを入力してください（空のまま Enter なら、ローカルの焼きなまし `neal` で解きます）。

In [ ]:
from getpass import getpass
token = getpass('Amplify のアクセストークン (使わない場合は空のまま Enter) ')
backend = 'amplify' if token else 'neal'
print('backend =', backend)

# 地域データを選ぶ

そのまま実行すると新緑県のデータを使います。別の地域の JSON を使う場合は、`use_upload = True` にしてアップロードしてください。

In [ ]:
use_upload = False
data_path = 'data/data_transformed.json'
if use_upload:
    from google.colab import files
    up = files.upload()
    data_path = list(up)[0]
print('地域データ:', data_path)

# シナリオ（機体・バッテリー・トラブル）

`data/scenario_example.json` を読み込み、ここで書き換えられます。書かなかった項目は既定値になります。

- `fleet`: センサー種別ごとの機体数・速度・見える範囲（m）・消費
- `battery`: `capacity_sec`（標準消費で飛べる秒数）、`full_charge_sec`（満充電にかかる秒数）
- `incidents`: トラブル。`hazard`（危険区域）・`survivor`（要救助者）は**ドローンが見つけるまで隠れています**。
  `drone_failure`・`battery_fault` は指定時刻に起きます（機体番号は 1 始まり）

In [ ]:
import json
scenario = json.load(open('data/scenario_example.json', encoding='utf-8'))
scenario['duration_sec'] = 21600                     # 運用時間 [s] (例は 6 時間. 計算に 15〜20 分)
# scenario['patrol'] = {'revisit_sec': 7200}         # 例: 2 時間ごとに巡回し直す
# scenario['launch_waves'] = 1                       # 1 にすると全機同時に出発 (例では 3 組で交代)
scenario['solver'] = {'backend': backend, 'time_limit_ms': 500}   # AE の1回あたりの制限時間 (毎秒に収まるように)
# 例: 1 つの市町村に最低 3 機置く
# scenario['deployment'] = {'min_per_region': 3}
# 例: 見える範囲や機体数を変える
# scenario['fleet'] = [{'sensor': 'SENS-BETA-02', 'count': 30, 'speed_mps': 15, 'sensor_radius_m': 200, 'drain_per_sec': 1.0}]
# 例: トラブルを足す
# scenario['incidents'].append({'type': 'drone_failure', 'time_sec': 900, 'drone': 3})
json.dump(scenario, open('my_scenario.json', 'w'), ensure_ascii=False, indent=1)
print(json.dumps(scenario['incidents'], ensure_ascii=False, indent=1))

# 実行（6 時間ぶんで 15〜20 分かかります）

In [ ]:
from realtime_drone import run
res, world, state = run(data_path, 'my_scenario.json', token=token or None)

# 動画

In [ ]:
from realtime_video import make_video
from IPython.display import Video
make_video(res, world, state, 'realtime_run.mp4')
Video('realtime_run.mp4', embed=True, width=1000)

# 記録

トラブルと、そのたびの QUBO 組み直し

In [ ]:
import pandas as pd
ev = pd.DataFrame(res['events'], columns=['t', 'kind', 'msg'])
ev['時刻'] = ev['t'].map(lambda t: f"{int(t)//60:02d}:{int(t)%60:02d}")
print('トラブル', len(ev), '件' if len(ev) else '件 (この時間内には起きませんでした)')
ev[['時刻', 'kind', 'msg']]

最適化の記録（毎秒の定期実行と、トラブル起因の組み直し）

In [ ]:
rp = pd.DataFrame(res['replans'])
print('最適化の回数', len(rp), ' / うちトラブル起因', (rp.trigger == 'event').sum())
print('1回あたり 組み立て', round(rp.build_ms.mean()), 'ms  求解', round(rp.solve_ms.mean()), 'ms (最大', round(rp.solve_ms.max()), 'ms)')
display(rp[rp.trigger == 'event'])
rp[['n_drones', 'n_vars', 'n_quadratic', 'build_ms', 'solve_ms']].describe().round(1)

配置の記録（配置 QUBO を解くたびの、市町村ごとの機数と配置換え）

In [ ]:
names = [m['name'].split(' (')[0] for m in res['municipalities']]
rows = []
for dp in res['deployments']:
    rows.append(dict(時刻=f"{int(dp['t'])//60:02d}:{int(dp['t'])%60:02d}", きっかけ=dp['trigger'],
                     **{n: c for n, c in zip(names, dp['counts'])},
                     配置換え=', '.join(f"機体{i+1} {names[a]}→{names[b]}" for i, a, b in dp['moved']) or '-',
                     求解ms=round(dp['solve_ms'])))
pd.DataFrame(rows)

いまの配置（機体ごとの担当市町村と基地）

In [ ]:
last = res['deployments'][-1]['assign']
pd.DataFrame([dict(機体=int(d)+1, センサー=res['drones'][int(d)]['sensor'], 担当=names[res['point_region'][p]],
                   基地x=round(res['points'][p][0]), 基地y=round(res['points'][p][1]))
              for d, p in sorted(last.items(), key=lambda kv: int(kv[0]))])

巡回の推移（一度でも巡回したブロックと、期限内に巡回できているブロックの割合）

In [ ]:
import matplotlib.pyplot as plt
tr = res['track']
plt.figure(figsize=(8, 3))
plt.plot([x['t']/3600 for x in tr], [x['visited'] for x in tr], label='一度でも巡回')
plt.plot([x['t']/3600 for x in tr], [x['fresh'] for x in tr], label='期限内')
plt.xlabel('時間 [h]'); plt.ylim(0, 1); plt.grid(alpha=.3); plt.legend(); plt.show()

市町村ごとの達成状況と総合スコア

In [ ]:
print('総合スコア', round(res['score'], 3), ' 要求達成', res['requirements_met'], '/', res['requirements_total'])
p = res['patrol']
print(f"巡回ブロック {p['blocks']} 個: 一度でも巡回 {p['visited']:.0%} / 期限内 ({p['revisit_sec']/60:.0f} 分以内) {p['fresh']:.0%} / 一度でも見たマス {p['cells_seen']:.1%}")
pd.DataFrame(res['municipalities'])

結果を保存

In [ ]:
from google.colab import files
json.dump(res, open('realtime_run.json', 'w'), ensure_ascii=False)
files.download('realtime_run.json')
files.download('realtime_run.mp4')